# Gemma 4 Vision Pipelines (Steps 21-24)

Gemma 4 is natively multimodal — it processes images and text together in a single
model call with no external OCR service. This notebook builds four practical vision
pipelines:

| Step | Pipeline | Use case |
|------|----------|----------|
| 21 | Receipt extractor | Parse line items, totals, dates from photos |
| 22 | Screenshot-to-data | Extract tables/UI data from screenshots |
| 23 | Document OCR + RAG | Index scanned PDFs/images, ask questions |
| 24 | Batch image analyser | Process folders of images at scale |

## What you need

**Local Ollama** (default — needs a desktop or GPU server):
```bash
pip install llama-index-llms-ollama llama-index-core llama-index-embeddings-huggingface
ollama pull gemma4:12b
```

**Ollama Cloud** (phone-only, no GPU needed):
```bash
pip install llama-index-llms-openai-like llama-index-core llama-index-embeddings-huggingface
export OLLAMA_CLOUD_API_KEY=sk-ollama-...   # from the Ollama app → Cloud API access
```
The first code cell auto-detects which mode to use based on `OLLAMA_CLOUD_API_KEY`.

In [ ]:
import base64
import json
import os
from pathlib import Path
from typing import Any

from llama_index.core.llms import ChatMessage
from llama_index.core.base.llms.types import ImageBlock, TextBlock

# ---------------------------------------------------------------------------
# LLM setup — auto-selects local Ollama or Ollama Cloud based on env vars
# Set OLLAMA_CLOUD_API_KEY (from the Ollama app) to use cloud mode.
# ---------------------------------------------------------------------------
OLLAMA_CLOUD_API_KEY = os.environ.get("OLLAMA_CLOUD_API_KEY", "")
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "")
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "gemma4:12b")
REQUEST_TIMEOUT = float(os.environ.get("OLLAMA_TIMEOUT", "180"))


def get_gemma_llm(model: str = OLLAMA_MODEL, timeout: float = REQUEST_TIMEOUT):
    if OLLAMA_CLOUD_API_KEY:
        from llama_index.llms.openai_like import OpenAILike
        base_url = OLLAMA_BASE_URL or "https://ollama.com/v1"
        print(f"Using Ollama CLOUD at {base_url} — model: {model}")
        return OpenAILike(
            model=model,
            api_base=base_url,
            api_key=OLLAMA_CLOUD_API_KEY,
            is_chat_model=True,
            is_function_calling_model=True,
            context_window=128_000,
            timeout=timeout,
        )
    else:
        from llama_index.llms.ollama import Ollama
        base_url = OLLAMA_BASE_URL or "http://localhost:11434"
        print(f"Using LOCAL Ollama at {base_url} — model: {model}")
        return Ollama(model=model, base_url=base_url, request_timeout=timeout)


llm = get_gemma_llm()


def load_image(path: str) -> bytes:
    with open(path, "rb") as f:
        return f.read()


def vision_chat(image_data: bytes, prompt: str) -> str:
    """Send an image + text prompt to Gemma 4 and return the text response."""
    response = llm.chat([
        ChatMessage(
            role="user",
            blocks=[
                ImageBlock(image=image_data),
                TextBlock(text=prompt),
            ],
        )
    ])
    return str(response.message.content)


print("Gemma 4 vision helper ready.")

## Step 21 — Receipt Extractor

Photograph a receipt and extract structured data: store name, date, line items with
prices, subtotal, tax, and total — output as JSON.

In [ ]:
RECEIPT_PROMPT = """Extract all information from this receipt image.
Return ONLY valid JSON with this schema:
{
  "store_name": str,
  "date": str (YYYY-MM-DD),
  "items": [{"name": str, "quantity": int, "unit_price": float, "total": float}],
  "subtotal": float,
  "tax": float,
  "total": float,
  "payment_method": str
}
If a field is not visible, use null."""


def extract_receipt(image_path: str) -> dict[str, Any]:
    image_data = load_image(image_path)
    raw = vision_chat(image_data, RECEIPT_PROMPT)
    # Strip markdown code fences if Gemma wraps the JSON
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(raw)


# Example — replace with your receipt image path
RECEIPT_PATH = os.environ.get("RECEIPT_IMAGE", "receipt.jpg")
if Path(RECEIPT_PATH).exists():
    result = extract_receipt(RECEIPT_PATH)
    print(json.dumps(result, indent=2))
    total_items = sum(item.get("total", 0) or 0 for item in result.get("items", []))
    print(f"\nExtracted {len(result.get('items', []))} line items, total: ${result.get('total')}")
else:
    print(f"Place a receipt image at '{RECEIPT_PATH}' or set RECEIPT_IMAGE env var.")

In [ ]:
# Batch receipt processing — process a whole folder
def batch_extract_receipts(folder: str) -> list[dict[str, Any]]:
    results = []
    for img_path in Path(folder).glob("*.{jpg,jpeg,png}"):
        print(f"Processing {img_path.name}...")
        try:
            data = extract_receipt(str(img_path))
            data["source_file"] = img_path.name
            results.append(data)
        except Exception as e:
            print(f"  Failed: {e}")
    return results


# Summarise multiple receipts
def summarise_receipts(receipts: list[dict[str, Any]]) -> None:
    total_spend = sum(r.get("total") or 0 for r in receipts)
    by_store: dict[str, float] = {}
    for r in receipts:
        store = r.get("store_name") or "Unknown"
        by_store[store] = by_store.get(store, 0) + (r.get("total") or 0)
    print(f"Total spend across {len(receipts)} receipts: ${total_spend:.2f}")
    print("By store:")
    for store, amount in sorted(by_store.items(), key=lambda x: -x[1]):
        print(f"  {store}: ${amount:.2f}")


RECEIPTS_FOLDER = os.environ.get("RECEIPTS_FOLDER", "receipts/")
if Path(RECEIPTS_FOLDER).is_dir():
    all_receipts = batch_extract_receipts(RECEIPTS_FOLDER)
    summarise_receipts(all_receipts)
else:
    print(f"Set RECEIPTS_FOLDER to a directory of receipt images to batch-process.")

## Step 22 — Screenshot-to-Data

Extract tables, form fields, UI data, and key metrics from screenshots — useful for
capturing data from apps that don't have APIs (bank statements, analytics dashboards).

In [ ]:
def extract_table_from_screenshot(image_path: str) -> str:
    """Extract any tabular data from a screenshot as Markdown."""
    image_data = load_image(image_path)
    return vision_chat(
        image_data,
        "Extract all tabular or structured data visible in this screenshot. "
        "Output as a Markdown table. If multiple tables exist, output all of them. "
        "If no table is visible, describe the key data points as a JSON object."
    )


def extract_form_fields(image_path: str) -> dict[str, str]:
    """Extract form field names and their values from a screenshot."""
    image_data = load_image(image_path)
    raw = vision_chat(
        image_data,
        "Extract all form fields visible in this screenshot. "
        "Return JSON: {\"field_name\": \"field_value\"}. "
        "For empty fields, use null as the value."
    )
    raw = raw.strip().lstrip("```json").rstrip("```")
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"raw_output": raw}


def describe_ui(image_path: str) -> str:
    """Describe the UI elements and user actions available on a screenshot."""
    image_data = load_image(image_path)
    return vision_chat(
        image_data,
        "Describe this UI screenshot: what app/page is shown, what are the main sections, "
        "what data is displayed, and what actions can the user take? Be concise."
    )


SCREENSHOT_PATH = os.environ.get("SCREENSHOT_IMAGE", "screenshot.png")
if Path(SCREENSHOT_PATH).exists():
    print("=== Table extraction ===")
    print(extract_table_from_screenshot(SCREENSHOT_PATH))
    print("\n=== UI description ===")
    print(describe_ui(SCREENSHOT_PATH))
else:
    print(f"Set SCREENSHOT_IMAGE env var to a screenshot path to test extraction.")

In [ ]:
# Capture key metrics from a dashboard screenshot
METRICS_PROMPT = """Identify all numeric metrics, KPIs, or statistics visible in this dashboard screenshot.
Return JSON: [{"metric_name": str, "value": str, "unit": str, "trend": "up"|"down"|"flat"|null}]
Include percentage changes if visible."""


def extract_dashboard_metrics(image_path: str) -> list[dict[str, Any]]:
    image_data = load_image(image_path)
    raw = vision_chat(image_data, METRICS_PROMPT)
    raw = raw.strip().lstrip("```json").rstrip("```")
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return [{"raw_output": raw}]


DASHBOARD_PATH = os.environ.get("DASHBOARD_IMAGE", "dashboard.png")
if Path(DASHBOARD_PATH).exists():
    metrics = extract_dashboard_metrics(DASHBOARD_PATH)
    print(json.dumps(metrics, indent=2))
else:
    print("Set DASHBOARD_IMAGE to a dashboard/analytics screenshot to extract KPIs.")

## Step 23 — Document OCR + RAG

Scan images of documents (PDFs exported as images, photos of letters, whiteboards)
into text, then index for semantic search. All local — sensitive documents never
leave your machine.

In [ ]:
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.storage import StorageContext
from llama_index.core import load_index_from_storage
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Fully local embeddings
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.llm = llm

OCR_PROMPT = """Transcribe all text visible in this document image exactly as written.
Preserve formatting where possible (paragraphs, bullet points, headings).
If there are handwritten sections, transcribe them too and mark with [handwritten].
Output only the transcribed text, nothing else."""


def ocr_image(image_path: str) -> str:
    """Transcribe text from a document image using Gemma 4 vision."""
    image_data = load_image(image_path)
    return vision_chat(image_data, OCR_PROMPT)


def build_ocr_index(image_paths: list[str], persist_dir: str = "./ocr_index") -> VectorStoreIndex:
    """OCR a list of document images and build a searchable RAG index."""
    documents = []
    for path in image_paths:
        print(f"OCR-ing {path}...")
        text = ocr_image(path)
        documents.append(Document(
            text=text,
            metadata={"source": path, "type": "ocr_document"},
        ))
        print(f"  Extracted {len(text)} chars.")

    index = VectorStoreIndex.from_documents(documents)
    index.storage_context.persist(persist_dir=persist_dir)
    print(f"\nIndex built and saved to {persist_dir}")
    return index


def load_ocr_index(persist_dir: str = "./ocr_index") -> VectorStoreIndex:
    storage_context = StorageContext.from_defaults(persist_dir=persist_dir)
    return load_index_from_storage(storage_context)


print("OCR + RAG functions ready.")

In [ ]:
# Build an index from a folder of scanned documents
DOCS_FOLDER = os.environ.get("SCANNED_DOCS_FOLDER", "scanned_docs/")
OCR_INDEX_DIR = "./ocr_index"

if Path(DOCS_FOLDER).is_dir():
    image_files = [
        str(p) for p in Path(DOCS_FOLDER).glob("*")
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
    ]
    print(f"Found {len(image_files)} document images.")
    ocr_index = build_ocr_index(image_files, persist_dir=OCR_INDEX_DIR)
elif Path(OCR_INDEX_DIR).exists():
    print("Loading existing OCR index...")
    ocr_index = load_ocr_index(OCR_INDEX_DIR)
else:
    print(f"Set SCANNED_DOCS_FOLDER to a directory of document images to build the index.")
    ocr_index = None

In [ ]:
# Query the OCR'd document index
if ocr_index:
    query_engine = ocr_index.as_query_engine(similarity_top_k=4)

    # Example queries — adapt to your documents
    queries = [
        "What is the total amount due on any invoices?",
        "What dates are mentioned in these documents?",
        "Are there any signatures or names mentioned?",
    ]
    for q in queries:
        print(f"\nQ: {q}")
        print(f"A: {query_engine.query(q)}")

In [ ]:
# Finance-specific OCR: bank statement analyser
STATEMENT_PROMPT = """This is a bank or credit card statement. Extract:
1. Account number (last 4 digits only)
2. Statement period (start and end dates)
3. Opening balance
4. Closing balance
5. All transactions: {date, description, amount (+ for credit, - for debit)}
Return as JSON. Redact full account numbers — keep only last 4 digits."""


def analyse_statement(image_path: str) -> dict[str, Any]:
    image_data = load_image(image_path)
    raw = vision_chat(image_data, STATEMENT_PROMPT)
    raw = raw.strip().lstrip("```json").rstrip("```")
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"raw_output": raw}


STATEMENT_PATH = os.environ.get("STATEMENT_IMAGE", "statement.jpg")
if Path(STATEMENT_PATH).exists():
    statement = analyse_statement(STATEMENT_PATH)
    print(json.dumps(statement, indent=2))
    if "transactions" in statement:
        debits = [t for t in statement["transactions"] if (t.get("amount") or 0) < 0]
        credits = [t for t in statement["transactions"] if (t.get("amount") or 0) > 0]
        print(f"\nTransactions: {len(debits)} debits, {len(credits)} credits")
else:
    print("Set STATEMENT_IMAGE to a bank statement photo to analyse.")

## Step 24 — Batch Image Analyser

Process an entire folder of images at scale — classify, describe, tag, or extract
structured data from each one. Useful for photo libraries, product catalogues,
document archives.

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor


IMAGE_CLASSIFY_PROMPT = """Classify this image. Return JSON:
{
  "category": str (receipt|document|screenshot|photo|chart|other),
  "description": str (1 sentence),
  "tags": [str] (up to 5 relevant tags),
  "text_present": bool,
  "confidence": "high"|"medium"|"low"
}"""


def classify_image(image_path: str) -> dict[str, Any]:
    image_data = load_image(image_path)
    raw = vision_chat(image_data, IMAGE_CLASSIFY_PROMPT)
    raw = raw.strip().lstrip("```json").rstrip("```")
    try:
        result = json.loads(raw)
        result["file"] = Path(image_path).name
        return result
    except json.JSONDecodeError:
        return {"file": Path(image_path).name, "raw": raw}


def batch_classify(folder: str, max_workers: int = 2) -> list[dict[str, Any]]:
    """Classify all images in a folder. max_workers=2 avoids overwhelming Ollama."""
    image_files = [
        str(p) for p in Path(folder).glob("*")
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".gif"}
    ]
    print(f"Classifying {len(image_files)} images...")
    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(classify_image, p): p for p in image_files}
        for future, path in futures.items():
            try:
                result = future.result(timeout=120)
                results.append(result)
                print(f"  {result['file']}: {result.get('category', '?')} — {result.get('description', '')[:60]}")
            except Exception as e:
                print(f"  {Path(path).name}: error — {e}")
    return results


IMAGES_FOLDER = os.environ.get("IMAGES_FOLDER", "images/")
if Path(IMAGES_FOLDER).is_dir():
    classifications = batch_classify(IMAGES_FOLDER)
    # Group by category
    from collections import Counter
    cats = Counter(r.get("category", "unknown") for r in classifications)
    print("\nBy category:", dict(cats))
else:
    print("Set IMAGES_FOLDER to a directory of images to run batch classification.")

In [ ]:
# Smart router: classify first, then apply the right extractor
def smart_extract(image_path: str) -> dict[str, Any]:
    """Auto-detect image type and apply the appropriate extractor."""
    classification = classify_image(image_path)
    category = classification.get("category", "other")
    print(f"Detected: {category} — {classification.get('description', '')}")

    if category == "receipt":
        return {"type": "receipt", "data": extract_receipt(image_path)}
    elif category == "screenshot":
        return {"type": "screenshot", "data": extract_table_from_screenshot(image_path)}
    elif category == "document":
        return {"type": "document", "data": ocr_image(image_path)}
    elif category == "chart":
        return {"type": "chart", "data": extract_dashboard_metrics(image_path)}
    else:
        return {"type": "other", "data": classification.get("description", "")}


# Example: drop any image and get auto-extracted data
ANY_IMAGE = os.environ.get("ANY_IMAGE", "")
if ANY_IMAGE and Path(ANY_IMAGE).exists():
    result = smart_extract(ANY_IMAGE)
    print(json.dumps(result, indent=2, default=str))
else:
    print("Set ANY_IMAGE to any image path — the smart router will classify and extract it.")

## Putting it all together: private finance assistant

Combine receipt OCR + statement analysis + RAG to build a complete personal
finance assistant that never sends your data to the cloud.

In [ ]:
from llama_index.core.agent.workflow import ReActAgent
from llama_index.core.tools import FunctionTool, QueryEngineTool


def extract_receipt_tool(image_path: str) -> str:
    """Extract structured data from a receipt image. Returns JSON string."""
    if not Path(image_path).exists():
        return f"File not found: {image_path}"
    result = extract_receipt(image_path)
    return json.dumps(result)


def ocr_document_tool(image_path: str) -> str:
    """Transcribe text from a document or letter image."""
    if not Path(image_path).exists():
        return f"File not found: {image_path}"
    return ocr_image(image_path)


def describe_image_tool(image_path: str) -> str:
    """Describe what is in any image — classify it and explain its contents."""
    if not Path(image_path).exists():
        return f"File not found: {image_path}"
    return json.dumps(classify_image(image_path))


vision_tools = [
    FunctionTool.from_defaults(fn=extract_receipt_tool),
    FunctionTool.from_defaults(fn=ocr_document_tool),
    FunctionTool.from_defaults(fn=describe_image_tool),
]

# Add the OCR index as a searchable tool if it was built
if 'ocr_index' in dir() and ocr_index is not None:
    vision_tools.append(QueryEngineTool.from_defaults(
        query_engine=ocr_index.as_query_engine(similarity_top_k=4),
        name="search_documents",
        description="Search previously OCR'd documents for specific information.",
    ))

finance_agent = ReActAgent(
    tools=vision_tools,
    llm=llm,
    max_iterations=10,
    verbose=True,
)

print(f"Finance vision agent ready with {len(vision_tools)} tools.")

In [ ]:
# Ask the agent to analyse images
# receipt_path = "grocery_receipt.jpg"  # your receipt photo
# response = await finance_agent.run(
#     f"Extract the receipt at '{receipt_path}'. "
#     "What did I spend the most on? Was there any tax charged?"
# )
# print(response)

## Privacy reminders

- All vision processing runs through **local Ollama** — images never leave your machine
- `BAAI/bge-small-en-v1.5` embeddings are cached locally after first download
- The OCR index is persisted locally in `./ocr_index` — keep this directory out of git
  (add it to `.gitignore`)
- Bank statements and financial documents are especially sensitive: run this notebook
  in a private environment, never on a shared machine
- The `analyse_statement` function redacts full account numbers — only the last 4 digits
  appear in extracted JSON